# Homework 3 - Star Schema and Interactive Dashboard for Financial Transactions

### Cristian Catania - 1000034809


In [41]:
# HOMEWORK #3 - PART 2.1 : LOAD AND CLEAN THE DATA
# Build the star-schema dataframes (4 dimensions + 1 fact) from the 3 raw CSVs,
# run the required data-quality checks, and keep only the modeled attributes.

import pandas as pd

UPLOADS = "."

# LOAD THE THREE RAW CSV FILES
# # The account statement and symbols files use ';' as separator and have a UTF-8
# BOM (utf-8-sig handles it). The country file uses ',' as separator.
# IDTransaction is read as string to avoid losing precision on the 10-digit IDs.

tx_raw = pd.read_csv(f"{UPLOADS}/account-statement-1-1-2024-12-31-2024.csv",
                     sep=";", encoding="utf-8-sig", dtype={"IDTransaction": str})
symbols = pd.read_csv(f"{UPLOADS}/symbols.csv", sep=";", encoding="utf-8-sig")
# country.csv: disable default NA parsing so Namibia's ISO code "NA" is NOT read as
# a missing value (classic pandas pitfall). Only truly empty cells become NaN.
country = pd.read_csv(f"{UPLOADS}/country.csv", sep=",", encoding="utf-8-sig", keep_default_na=False, na_values=[""])

print("RAW SHAPES -> tx:", tx_raw.shape, "| symbols:", symbols.shape, "| country:", country.shape)

RAW SHAPES -> tx: (2745, 6) | symbols: (3194, 5) | country: (249, 11)


In [42]:
# - CLEAN THE ACCOUNT STATEMENT

tx = tx_raw.copy()

#  1a. Drop the phantom column created by the trailing ';' in every line.
if "Unnamed: 5" in tx.columns:
    tx = tx.drop(columns=["Unnamed: 5"])

# 1b. Drop fully-empty rows (the file has 464 blank rows at the bottom).
tx = tx.dropna(how="all").reset_index(drop=True)

# 1c. Normalize the transaction type: the source has a typo "DIVIDENT" -> "DIVIDEND".
tx["TransactionType"] = tx["TransactionType"].str.strip().replace({"DIVIDENT": "DIVIDEND"})

# 1d. Parse the Date string (dd/mm/YYYY HH:MM:SS) into a real datetime,
# then derive the time attributes required by Dim_Time.
tx["Date"] = pd.to_datetime(tx["Date"], format="%d/%m/%Y %H:%M:%S")
tx["date"]        = tx["Date"].dt.date                 # day-level date (grain of Dim_Time)
tx["day_of_week"] = tx["Date"].dt.day_name()           # e.g. "Monday" (needed by Q12)
tx["month"]       = tx["Date"].dt.month
tx["quarter"]     = tx["Date"].dt.quarter
tx["year"]        = tx["Date"].dt.year

# 1e. Cast the measure to integer.
tx["Unit"] = tx["Unit"].astype(int)

print("CLEANED TX SHAPE:", tx.shape)
print("Transaction types:", tx["TransactionType"].unique().tolist())

CLEANED TX SHAPE: (2281, 10)
Transaction types: ['BUY', 'SELL', 'DIVIDEND']


In [43]:
# - DATA QUALITY CHECK : MISSING VALUES (on all three source dataframes)

print("\n[DQ-1a] Missing values in cleaned transactions:")
print(tx[["IDTransaction","Date","TransactionType","Symbol","Unit"]].isnull().sum().to_string())
 
print("\n[DQ-1b] Missing values in symbols.csv:")
print(symbols.isnull().sum().to_string())
 
print("\n[DQ-1c] Missing values in country.csv (only the modeled columns):")
print(country[["name","alpha-2","sub-region","region"]].isnull().sum().to_string())

# Surface WHICH countries have a NULL region/sub-region (found: Antarctica, Taiwan).
# Antarctica is never traded, so it is irrelevant; Taiwan is fixed in STEP 5.
missing_geo = country[country["region"].isnull() | country["sub-region"].isnull()]["name"].tolist()
print("        Countries with NULL region/sub-region:", missing_geo)


[DQ-1a] Missing values in cleaned transactions:
IDTransaction      0
Date               0
TransactionType    0
Symbol             0
Unit               0

[DQ-1b] Missing values in symbols.csv:
symbol          0
company_name    0
sector          0
industry        0
country         0

[DQ-1c] Missing values in country.csv (only the modeled columns):
name          0
alpha-2       0
sub-region    2
region        2
        Countries with NULL region/sub-region: ['Antarctica', 'Taiwan, Province of China']


In [44]:
# - DATA QUALITY CHECK : every transaction symbol must exist in symbols.csv
#
# Build a set with the distinct tickers that actually appear in the transactions.
tx_symbols  = set(tx["Symbol"].unique())
# Build a set with all tickers available in the symbols reference (the catalogue).
ref_symbols = set(symbols["symbol"].unique())
# Set difference: tickers traded but NOT present in the catalogue (the "orphans").
missing_symbols = sorted(tx_symbols - ref_symbols)
# Count how many transaction ROWS use one of those orphan tickers.
orphan_rows = tx["Symbol"].isin(missing_symbols).sum()

# Report: how many distinct tickers are traded and how many cannot be enriched.
print(f"\n[DQ-2] Symbols traded: {len(tx_symbols)} | missing from symbols.csv: {len(missing_symbols)}")
# Report: absolute number and percentage of transactions left without enrichment.
print(f"        -> {orphan_rows} transactions ({100*orphan_rows/len(tx):.1f}%) have no enrichment")
# Report: the actual list of orphan tickers, for documentation in the report.
print("        Missing tickers:", missing_symbols)


[DQ-2] Symbols traded: 129 | missing from symbols.csv: 18
        -> 212 transactions (9.3%) have no enrichment
        Missing tickers: ['AGO.l', 'ARCH', 'AZM', 'CCAP', 'CSIQ', 'FNC', 'HTGC', 'IBE', 'MFG', 'MONC', 'OBDC', 'RCMT', 'RIGZU', 'SAP', 'TKC', 'UCG', 'VWS', 'WF']


In [45]:
# - DATA QUALITY CHECK : every company country must map to country.csv
# symbols.csv uses short names ("Turkey", "Taiwan") that differ from the
# official ISO names in country.csv. We define an explicit mapping instead of
# losing those rows.
#
# Manual name fixes: source name (symbols.csv) -> official name (country.csv)
COUNTRY_NAME_FIX = {
    "Turkey": "Türkiye",
    "Taiwan": "Taiwan, Province of China",
}

# Check which symbol countries cannot be matched BEFORE applying the fix.
# Build a set with the distinct country names used in the symbols catalogue.
sym_countries = set(symbols["country"].unique())
# Build a set with the official country names available in country.csv.
ref_country_names = set(country["name"].unique())
# Set difference: country names present in symbols.csv but NOT found in country.csv.
# This is checked before applying the fix, to expose the naming mismatches.
unmatched_before = sorted(sym_countries - ref_country_names)

# Report the country names that do not match directly (found: "Turkey", "Taiwan").
print("\n[DQ-3] Countries in symbols.csv not directly in country.csv:", unmatched_before)
# Report how they are resolved: an explicit mapping to the official ISO names.
print("        -> resolved via explicit name mapping:", COUNTRY_NAME_FIX)


[DQ-3] Countries in symbols.csv not directly in country.csv: ['Taiwan', 'Turkey']
        -> resolved via explicit name mapping: {'Turkey': 'Türkiye', 'Taiwan': 'Taiwan, Province of China'}


In [46]:
# - BUILD THE GEOGRAPHY REFERENCE (country -> sub_region -> region)
# Keep ONLY the attributes used in our dimensional model: name, alpha-2,
# sub-region, region. All other columns (alpha-3, country-code, iso_3166-2,
# intermediate-region, *-code) are dropped as required ("remove unused attributes").

geo_ref = country[["name", "alpha-2", "sub-region", "region"]].copy()
geo_ref = geo_ref.rename(columns={"name": "country",
                                  "alpha-2": "alpha_2",
                                  "sub-region": "sub_region"})
 
# Fix Taiwan: in country.csv its region/sub-region are NULL. Fill them manually.
geo_ref.loc[geo_ref["country"] == "Taiwan, Province of China",
            ["sub_region", "region"]] = ["Eastern Asia", "Asia"] 

In [47]:
# - ENRICH THE TRANSACTIONS (symbol -> sector/industry/country -> geography)
# We use LEFT joins so that no transaction is ever dropped; missing values are
# filled with the "Unknown" member.
#
# Apply the country name fix on the symbols reference, then join symbol info.
sym_ref = symbols.copy()
sym_ref["country"] = sym_ref["country"].replace(COUNTRY_NAME_FIX)
tx_enr = tx.merge(sym_ref[["symbol", "company_name", "sector", "industry", "country"]],
                  left_on="Symbol", right_on="symbol", how="left")

# Drop the redundant join key: "Symbol" already holds the ticker, "symbol" is its duplicate.
tx_enr = tx_enr.drop(columns=["symbol"])

# Attach geography (sub_region, region, alpha_2) via the company country.
tx_enr = tx_enr.merge(geo_ref, on="country", how="left")

# Fill every missing descriptive attribute with "Unknown" so totals stay correct.
for col in ["company_name", "sector", "industry", "country", "alpha_2", "sub_region", "region"]:
    tx_enr[col] = tx_enr[col].fillna("Unknown")
 
print("\nENRICHED TX SHAPE:", tx_enr.shape, "| rows preserved:", len(tx_enr) == len(tx))


ENRICHED TX SHAPE: (2281, 17) | rows preserved: True


In [48]:
# - BUILD THE DIMENSION TABLES (with integer surrogate keys)
# Each dimension keeps ONLY the attributes defined in the star schema.
#
# Dim_Time : one row per distinct calendar day actually present in the data.
dim_time = (tx_enr[["date", "day_of_week", "month", "quarter", "year"]]
            .drop_duplicates()
            .sort_values("date")
            .reset_index(drop=True))
dim_time.insert(0, "time_id", range(1, len(dim_time) + 1))   # surrogate key

# Dim_Symbol : one row per distinct traded ticker (enrichment may be Unknown).
dim_symbol = (tx_enr[["Symbol", "company_name", "sector", "industry"]]
              .drop_duplicates()
              .rename(columns={"Symbol": "symbol"})
              .sort_values("symbol")
              .reset_index(drop=True))
dim_symbol.insert(0, "symbol_id", range(1, len(dim_symbol) + 1))

# Dim_Geography : one row per distinct country (country -> sub_region -> region).
dim_geography = (tx_enr[["country", "alpha_2", "sub_region", "region"]]
                 .drop_duplicates()
                 .sort_values("country")
                 .reset_index(drop=True))
dim_geography.insert(0, "geo_id", range(1, len(dim_geography) + 1))

# Dim_TransactionType : one row per transaction type (BUY / SELL / DIVIDEND).
dim_txtype = (tx_enr[["TransactionType"]]
              .drop_duplicates()
              .rename(columns={"TransactionType": "transaction_type"})
              .sort_values("transaction_type")
              .reset_index(drop=True))
dim_txtype.insert(0, "type_id", range(1, len(dim_txtype) + 1))

# Print the shape (rows, columns) of each dimension as a quick sanity check:
# confirms how many distinct members each dimension ended up with.
print("\nDIMENSION SHAPES -> Time:", dim_time.shape, "| Symbol:", dim_symbol.shape,
      "| Geography:", dim_geography.shape, "| TxType:", dim_txtype.shape)


DIMENSION SHAPES -> Time: (231, 6) | Symbol: (129, 5) | Geography: (22, 5) | TxType: (3, 2)


In [49]:
# - BUILD THE FACT TABLE (resolve each natural key into its surrogate FK)

fact = tx_enr.copy()

# Resolve the four foreign keys: for each transaction we look up the matching
# dimension row and bring back ONLY its surrogate key (the *_id column).

# Look up the day in Dim_Time -> attach time_id (join on the shared "date" column).
fact = fact.merge(dim_time[["time_id", "date"]], on="date", how="left")
# Look up the ticker in Dim_Symbol -> attach symbol_id.
# Column names differ ("Symbol" here vs "symbol" there), so we use left_on/right_on.
fact = fact.merge(dim_symbol[["symbol_id", "symbol"]],
                  left_on="Symbol", right_on="symbol", how="left")
# Look up the country in Dim_Geography -> attach geo_id (join on the shared "country").
fact = fact.merge(dim_geography[["geo_id", "country"]], on="country", how="left")
# Look up the type in Dim_TransactionType -> attach type_id (column names differ again).
fact = fact.merge(dim_txtype[["type_id", "transaction_type"]],
                  left_on="TransactionType", right_on="transaction_type", how="left")

# Reduce to the final fact columns only.
# Rename the measure from the source name "Unit" to the model name "units".
fact = fact.rename(columns={"Unit": "units"})
# Keep ONLY the fact columns: the 4 foreign keys, the measure, and the
# degenerate dimension (IDTransaction). All descriptive attributes stay in the dims.
fact = fact[["time_id", "symbol_id", "geo_id", "type_id", "units", "IDTransaction"]].copy()
# Add the surrogate primary key: a progressive integer 1..N, one per transaction.
fact.insert(0, "transaction_sk", range(1, len(fact) + 1))

# Sanity check: (rows, columns) of the final fact table.
print("\nFACT SHAPE:", fact.shape)


FACT SHAPE: (2281, 7)


In [50]:
# - FINAL INTEGRITY CHECKS
# Three sanity checks confirming the star schema was built correctly.
#
# Check 1: no NULL foreign keys. Every fact row must link to all 4 dimensions.
# .isnull() flags empty cells, the first .sum() counts them per column,
# the second .sum() totals across columns. Expected result: 0.
print("\n[INTEGRITY] Null foreign keys in fact:",
      fact[["time_id", "symbol_id", "geo_id", "type_id"]].isnull().sum().sum())

# Check 2: no rows lost or duplicated. The fact must have exactly as many rows
# as the cleaned transactions (the merges must not drop or multiply rows).
# Expected result: True.
print("[INTEGRITY] Fact rows == cleaned transactions:", len(fact) == len(tx))

# Check 3: the primary key is truly unique (one row per transaction).
# Expected result: True.
print("[INTEGRITY] transaction_sk unique:", fact["transaction_sk"].is_unique)


[INTEGRITY] Null foreign keys in fact: 0
[INTEGRITY] Fact rows == cleaned transactions: True
[INTEGRITY] transaction_sk unique: True


In [52]:
# HOMEWORK #3 - PART 2.2 : ANALYTICAL QUERIES
# We answer 5 of the 14 proposed questions (Q1, Q3, Q7, Q9, Q12), chosen so that
# together they exercise every dimension of the star schema.
#
pd.set_option("display.width", 200)
#
# STAR JOIN : denormalize the fact with all four dimensions ONCE.
# Each query below is then a simple filter + group-by on this single dataframe.
#
df = (fact
      .merge(dim_time,      on="time_id")      # adds date, day_of_week, month, quarter, year
      .merge(dim_symbol,    on="symbol_id")    # adds symbol, company_name, sector, industry
      .merge(dim_geography, on="geo_id")       # adds country, alpha_2, sub_region, region
      .merge(dim_txtype,    on="type_id"))     # adds transaction_type
 
# NOTE on "Unknown": the 18 orphan tickers carry "Unknown" sector/industry/country/
# region. We exclude "Unknown" only when ranking by an enrichment attribute, since
# the value is not derivable from the provided reference data. DIVIDEND rows are
# excluded from all queries (the questions concern BUY/SELL trading activity only).
#
# Q1 - Top 5 sectors by number of SELL transactions in the US during 2024.
#
q1 = df[(df["transaction_type"] == "SELL")
        & (df["country"] == "United States of America")
        & (df["year"] == 2024)
        & (df["sector"] != "Unknown")]
q1 = (q1.groupby("sector").size()                 # count transactions per sector
        .sort_values(ascending=False).head(5)
        .rename("sell_transactions").reset_index())
print("\n=== Q1: Top 5 sectors by SELL transactions in US (2024) ===")
print(q1.to_string(index=False))


=== Q1: Top 5 sectors by SELL transactions in US (2024) ===
                sector  sell_transactions
            Technology                158
Communication Services                 58
    Financial Services                 55
            Healthcare                 50
     Consumer Cyclical                 48


In [53]:
# Q3 - Rank all quarters of 2024 by total number of transactions (BUY + SELL).
#
q3 = df[(df["transaction_type"].isin(["BUY", "SELL"]))
        & (df["year"] == 2024)]
q3 = (q3.groupby("quarter").size()
        .sort_values(ascending=False)
        .rename("total_transactions").reset_index())
print("\n=== Q3: Quarters of 2024 ranked by total transactions (BUY+SELL) ===")
print(q3.to_string(index=False))


=== Q3: Quarters of 2024 ranked by total transactions (BUY+SELL) ===
 quarter  total_transactions
       1                1076
       2                 580
       3                 260
       4                 255


In [54]:
# Q7 - Top 10 symbols by number of transactions (BUY + SELL) in 2024.
# (symbol is never "Unknown", so no exclusion is needed here.)
#
q7 = df[(df["transaction_type"].isin(["BUY", "SELL"]))
        & (df["year"] == 2024)]
q7 = (q7.groupby("symbol").size()
        .sort_values(ascending=False).head(10)
        .rename("total_transactions").reset_index())
print("\n=== Q7: Top 10 symbols by total transactions (BUY+SELL, 2024) ===")
print(q7.to_string(index=False))


=== Q7: Top 10 symbols by total transactions (BUY+SELL, 2024) ===
symbol  total_transactions
   ARM                 100
   AMD                  97
   TSM                  77
  TIMB                  76
   FNC                  66
  GOOG                  48
  AMZN                  47
  MSFT                  45
  ARDX                  43
  BRFS                  42


In [55]:
# Q9 - Top 5 industries by units traded in Asia in 2024.
#
q9 = df[(df["region"] == "Asia")
        & (df["transaction_type"].isin(["BUY", "SELL"]))
        & (df["year"] == 2024)
        & (df["industry"] != "Unknown")]
q9 = (q9.groupby("industry")["units"].sum()       # sum of units traded per industry
        .sort_values(ascending=False).head(5)
        .rename("units_traded").reset_index())
print("\n=== Q9: Top 5 industries by units traded in Asia (2024) ===")
print(q9.to_string(index=False))


=== Q9: Top 5 industries by units traded in Asia (2024) ===
                      industry  units_traded
                Semiconductors          5401
            Auto Manufacturers          4769
        Diagnostics & Research          1945
Internet Content & Information          1805
           Aerospace & Defense          1799


In [56]:
# Q12 - Top 3 sectors by total units SOLD on Mondays across 2024.
# (no geographic filter here, so the "Unknown" exclusion is essential.)
#
q12 = df[(df["transaction_type"] == "SELL")
         & (df["day_of_week"] == "Monday")
         & (df["year"] == 2024)
         & (df["sector"] != "Unknown")]
q12 = (q12.groupby("sector")["units"].sum()
         .sort_values(ascending=False).head(3)
         .rename("units_sold").reset_index())
print("\n=== Q12: Top 3 sectors by units sold on Mondays (2024) ===")
print(q12.to_string(index=False))


=== Q12: Top 3 sectors by units sold on Mondays (2024) ===
           sector  units_sold
       Technology        4361
       Healthcare        2108
Consumer Cyclical        1218
